In [ ]:
# ==============================================================================
# 1. INSTALAÇÃO DE PACOTES E COMPATIBILIDADE
# ==============================================================================
print("⏳ Removendo versões antigas e instalando pacotes necessários...")

!pip uninstall -y whisper openai-whisper
!pip install -q -U openai-whisper
!pip install -q -U pyannote.audio
!pip install -q -U speechbrain
!pip install -q -U soundfile
!pip install -q -U einops
!pip install -q -U pydub
!apt-get update && apt-get install -y ffmpeg

# Força a versão do NumPy compatível com o numba/whisper por baixo dos panos
!pip install -q "numpy<2.1"

print("✅ Dependências instaladas com sucesso!")

In [ ]:
# ==============================================================================
# 2. ACOPLAMENTO DO GOOGLE DRIVE
# ==============================================================================
from google.colab import drive
import os

print("🔄 Conectando ao Google Drive (pode solicitar permissão na tela)...")
drive.mount('/content/drive', force_remount=True)

AUDIO_ORIGINAL = "/content/drive/MyDrive/Training/REC6.mp4"

print("\n--- TESTE DE CONEXÃO ---")
if os.path.exists(AUDIO_ORIGINAL):
    print("✅ Sucesso: Arquivo original encontrado no Drive!")
else:
    print("❌ Erro: Arquivo NÃO encontrado. Verifique se o caminho no seu Drive está correto.")

In [ ]:
# ==============================================================================
# 3. CRIAÇÃO DO RECORTE DE TESTE RÁPIDO
# ==============================================================================
AUDIO_ORIGINAL = "/content/drive/MyDrive/Training/REC6.mp4"
CLIP_TESTE = "/content/REC6_teste_120s.mp4"

print("⏳ Cortando os primeiros 120 segundos do áudio original...")
!ffmpeg -y -i "{AUDIO_ORIGINAL}" -t 120 -c copy "{CLIP_TESTE}"

import os
if os.path.exists(CLIP_TESTE):
    print("\n✅ Clipe de teste gerado com sucesso localmente em:", CLIP_TESTE)
else:
    print("\n❌ Falha ao gerar o clipe de teste.")

In [ ]:
# ==============================================================================
# 4. CONFIGURAÇÕES GLOBAIS E AUTENTICAÇÃO HUGGING FACE
# ==============================================================================
import os
import gc
import time
import json
import numpy as np
import torch
from google.colab import userdata
from huggingface_hub import login

# ⚠️ CONTROLE DE AMBIENTE: Mude para False somente após validar o teste de 120s!
MODO_TESTE = False

AUDIO_ORIGINAL = "/content/drive/MyDrive/Training/REC6.mp4"
CLIP_TESTE = "/content/REC6_teste_120s.mp4"
ARQUIVO_SAIDA = "/content/drive/MyDrive/Training/REC6.txt"
CHECKPOINT_FILE = "/content/drive/MyDrive/Training/REC6_checkpoint.json"

# Overlap entre chunks (em segundos): dá contexto extra pro Whisper/pyannote
# nas bordas de cada pedaço, evitando cortar uma palavra ou fala no meio.
OVERLAP_S = 3

# Limiar de similaridade de voz (0 a 1) para considerar que duas vozes em
# chunks diferentes são a mesma pessoa. Ajuste se dois participantes
# estiverem sendo unificados (aumente) ou o mesmo participante estiver
# sendo dividido em vários rótulos (diminua).
SIMILARITY_THRESHOLD = 0.75

# Decide os parâmetros com base no modo selecionado
if MODO_TESTE:
    print("🚀 [CONFIG] MODO DE TESTE ATIVADO: Processando apenas o recorte de 120s.")
    AUDIO_FILE = CLIP_TESTE
    CHUNK_LENGTH_MS = 2 * 60 * 1000  # Chunks de 2 minutos
else:
    print("🔥 [CONFIG] MODO PRODUÇÃO ATIVADO: Processando o arquivo completo de 3h.")
    AUDIO_FILE = AUDIO_ORIGINAL
    CHUNK_LENGTH_MS = 20 * 60 * 1000  # Chunks de 20 minutos

# Autenticação Hugging Face para o Pyannote
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# Identificação de Placa de Vídeo (GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🤖 Hardware alocado para processamento: {device}")

## 5. Carregamento dos modelos (uma única vez) + funções de apoio

Antes rodávamos `Pipeline.from_pretrained(...)` e `whisper.load_model(...)` **dentro** do
loop, recarregando os modelos a cada chunk. Agora carregamos tudo uma vez só, aqui.

Também carregamos um modelo de *speaker embedding* (ECAPA-TDNN via SpeechBrain) para
conseguir reconhecer a mesma pessoa entre chunks diferentes — sem isso, os rótulos
SPEAKER_00/01 do pyannote só valem *dentro* de cada chunk, não ao longo da call inteira.

In [ ]:
# ==============================================================================
# 5. CARREGAMENTO DOS MODELOS (UMA ÚNICA VEZ) + FUNÇÕES DE APOIO
# ==============================================================================
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
import whisper
import torchaudio

try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("⏳ Carregando modelo de diarização (pyannote)...")
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)
diarization_pipeline.to(device)

print("⏳ Carregando modelo Whisper...")
whisper_model = whisper.load_model("small", device=device)

print("⏳ Carregando modelo de speaker embedding (ECAPA-TDNN)...")
embedding_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained_models/spkrec-ecapa-voxceleb",
    run_opts={"device": device.type}
)

print("✅ Todos os modelos carregados (só serão usados no loop, não recarregados).")


# --------------------------------------------------------------------------
# Funções de apoio para reconhecer o mesmo falante entre chunks diferentes
# --------------------------------------------------------------------------

def extrair_embedding(wav_path, start_s, end_s):
    """Extrai o vetor de voz (embedding) de um trecho de áudio."""
    waveform, sr = torchaudio.load(wav_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
        sr = 16000

    start_sample = max(0, int(start_s * sr))
    end_sample = min(waveform.shape[1], int(end_s * sr))

    if end_sample - start_sample < int(0.3 * sr):  # trecho curto demais (<0.3s)
        return None

    segmento = waveform[:, start_sample:end_sample]

    with torch.no_grad():
        emb = embedding_model.encode_batch(segmento)

    return emb.squeeze().detach().cpu().numpy()


def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


def mapear_falante_global(embedding, global_speakers):
    """Compara o embedding com falantes globais já vistos.
    Se for parecido o suficiente (>= SIMILARITY_THRESHOLD), reaproveita o rótulo
    global existente. Senão, cria um novo participante global."""
    melhor_gs = None
    melhor_sim = -1.0

    for gs in global_speakers:
        sim = cosine_sim(embedding, gs["embedding"])
        if sim > melhor_sim:
            melhor_sim = sim
            melhor_gs = gs

    if melhor_gs is not None and melhor_sim >= SIMILARITY_THRESHOLD:
        # Atualiza a média do embedding desse participante (fica mais robusto com o tempo)
        n = melhor_gs["count"]
        melhor_gs["embedding"] = (melhor_gs["embedding"] * n + embedding) / (n + 1)
        melhor_gs["count"] += 1
        return melhor_gs["label"]
    else:
        novo_label = f"PARTICIPANTE_{len(global_speakers) + 1}"
        global_speakers.append({"label": novo_label, "embedding": embedding, "count": 1})
        return novo_label


# --------------------------------------------------------------------------
# Checkpoint: retoma de onde parou se o notebook foi reiniciado/desconectado
# --------------------------------------------------------------------------

def salvar_checkpoint(proximo_chunk_ms, all_diar, all_trans, global_speakers):
    checkpoint = {
        "proximo_chunk_ms": proximo_chunk_ms,
        "all_diarization_segments": all_diar,
        "all_transcription_segments": all_trans,
        "global_speakers": [
            {"label": gs["label"], "embedding": gs["embedding"].tolist(), "count": gs["count"]}
            for gs in global_speakers
        ],
    }
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f)


if os.path.exists(CHECKPOINT_FILE):
    print("🔄 Checkpoint encontrado! Retomando de onde parou...")
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)
    start_chunk_ms = checkpoint["proximo_chunk_ms"]
    all_diarization_segments = checkpoint["all_diarization_segments"]
    all_transcription_segments = checkpoint["all_transcription_segments"]
    global_speakers = [
        {"label": gs["label"], "embedding": np.array(gs["embedding"]), "count": gs["count"]}
        for gs in checkpoint["global_speakers"]
    ]
    print(f"   Retomando a partir de {start_chunk_ms/1000:.1f}s, com {len(global_speakers)} participante(s) já identificado(s).")
else:
    start_chunk_ms = 0
    all_diarization_segments = []
    all_transcription_segments = []
    global_speakers = []
    print("ℹ️ Nenhum checkpoint encontrado, começando do zero.")

## 6. Execução em loop (diarização + Whisper), com contexto de overlap e checkpoint

Diferenças da versão anterior:
- Modelos **não** são recarregados a cada chunk (usa os já carregados na célula acima).
- Cada chunk é exportado como **.wav** (evita reencodar em AAC/mp4 sem necessidade).
- Cada chunk inclui uma pequena margem de **overlap** (`OVERLAP_S`) nas bordas para não
  cortar uma palavra/fala no meio — a margem é descartada dos resultados finais, só serve
  de contexto extra para os modelos.
- Os rótulos de falante são convertidos para um rótulo **global** (`PARTICIPANTE_1`,
  `PARTICIPANTE_2`, ...) usando o embedding de voz, então a mesma pessoa recebe o mesmo
  rótulo em qualquer chunk.
- Depois de cada chunk, um **checkpoint** é salvo no Drive. Se o Colab desconectar no meio
  do processamento, basta rodar tudo de novo — ele retoma de onde parou.

In [ ]:
# ==============================================================================
# 6. EXECUÇÃO EM LOOP (DIARIZAÇÃO + WHISPER)
# ==============================================================================
from pydub import AudioSegment

print("🎵 Carregando arquivo de áudio na memória...")
audio = AudioSegment.from_file(AUDIO_FILE)
duration_ms = len(audio)

for chunk_start_ms in range(start_chunk_ms, duration_ms, CHUNK_LENGTH_MS):
    chunk_end_ms = min(chunk_start_ms + CHUNK_LENGTH_MS, duration_ms)

    # Janela exportada = chunk + margem de overlap nas bordas (contexto extra)
    export_start_ms = max(0, chunk_start_ms - OVERLAP_S * 1000)
    export_end_ms = min(duration_ms, chunk_end_ms + OVERLAP_S * 1000)

    print("\n" + "=" * 60)
    print(f"📦 TRABALHANDO NO CHUNK: {chunk_start_ms / 1000:.1f}s até {chunk_end_ms / 1000:.1f}s")
    print("=" * 60)

    chunk_file = f"/tmp/chunk_{chunk_start_ms}.wav"
    chunk_audio = audio[export_start_ms:export_end_ms]
    chunk_audio.export(chunk_file, format="wav")

    # --- PASSO 1: DIARIZAÇÃO ---
    print("\n[1/3] Identificando troca de vozes (Pyannote 3.1)...")
    t0 = time.time()

    with torch.no_grad():
        with ProgressHook() as hook:
            diarization_chunk = diarization_pipeline(chunk_file, hook=hook)

    print(f"✅ Diarização do chunk finalizada em {time.time() - t0:.1f}s")

    if hasattr(diarization_chunk, "speaker_diarization"):
        annotation_data = diarization_chunk.speaker_diarization
    elif hasattr(diarization_chunk, "annotation"):
        annotation_data = diarization_chunk.annotation
    else:
        annotation_data = diarization_chunk

    # Agrupa turnos por rótulo LOCAL (válido só dentro deste chunk)
    turnos_por_falante_local = {}
    todos_os_turnos = list(annotation_data.itertracks(yield_label=True))
    for turn, _, speaker_local in todos_os_turnos:
        turnos_por_falante_local.setdefault(speaker_local, []).append(turn)

    # Para cada falante local, pega o turno mais longo (voz mais "limpa") e
    # extrai um embedding para descobrir quem é essa pessoa globalmente
    mapa_local_para_global = {}
    for speaker_local, turnos in turnos_por_falante_local.items():
        turno_mais_longo = max(turnos, key=lambda t: t.end - t.start)
        embedding = extrair_embedding(chunk_file, turno_mais_longo.start, turno_mais_longo.end)

        if embedding is not None:
            label_global = mapear_falante_global(embedding, global_speakers)
        else:
            label_global = f"PARTICIPANTE_DESCONHECIDO_{speaker_local}"

        mapa_local_para_global[speaker_local] = label_global

    print("   Falantes neste chunk:", mapa_local_para_global)

    # Guarda os turnos de diarização já com rótulo GLOBAL e no tempo absoluto do
    # áudio inteiro. Descarta a margem de overlap (só serve de contexto).
    for turn, _, speaker_local in todos_os_turnos:
        abs_start = turn.start + (export_start_ms / 1000)
        abs_end = turn.end + (export_start_ms / 1000)

        if abs_start < chunk_start_ms / 1000 or abs_start >= chunk_end_ms / 1000:
            continue  # está na margem de overlap, ignora pra não duplicar

        all_diarization_segments.append((abs_start, abs_end, mapa_local_para_global[speaker_local]))

    # --- PASSO 2: TRANSCRIÇÃO ---
    print("\n[2/3] Transcrevendo áudio em texto (Whisper)...")
    t0 = time.time()

    with torch.no_grad():
        result_chunk = whisper_model.transcribe(
            chunk_file,
            language="en",
            task="transcribe",
            fp16=torch.cuda.is_available(),
            verbose=False
        )

    print(f"✅ Transcrição do chunk finalizada em {time.time() - t0:.1f}s")

    for seg in result_chunk["segments"]:
        abs_start = seg["start"] + export_start_ms / 1000
        abs_end = seg["end"] + export_start_ms / 1000

        if abs_start < chunk_start_ms / 1000 or abs_start >= chunk_end_ms / 1000:
            continue  # está na margem de overlap, ignora pra não duplicar

        seg["start"] = abs_start
        seg["end"] = abs_end
        all_transcription_segments.append(seg)

    # Limpa memória e apaga o arquivo temporário do chunk
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    os.remove(chunk_file)

    # Salva checkpoint: se cair aqui, o próximo run retoma deste ponto em diante
    salvar_checkpoint(chunk_end_ms, all_diarization_segments, all_transcription_segments, global_speakers)

print("\n\n🏆 FIM DO PROCESSAMENTO: Todos os pedaços foram processados com sucesso!")
print(f"👥 Total de participantes distintos identificados: {len(global_speakers)}")

In [ ]:
# ==============================================================================
# 7. CRUZA DADOS (ÁUDIO + TEXTO) E SALVA NO DRIVE
# ==============================================================================
print("\n[3/3] Iniciando cruzamento inteligente de falantes com textos...")

resultado = []

for seg in all_transcription_segments:
    start_time = seg["start"]
    end_time = seg["end"]
    text = seg["text"].strip()

    overlaps = []

    for diar_start, diar_end, speaker in all_diarization_segments:
        overlap = max(0, min(end_time, diar_end) - max(start_time, diar_start))
        if overlap > 0:
            overlaps.append((overlap, speaker))

    if overlaps:
        speaker_final = max(overlaps, key=lambda x: x[0])[1]
    else:
        speaker_final = "UNKNOWN"

    linha = f"[{start_time:.1f}s - {end_time:.1f}s] {speaker_final}: {text}"
    resultado.append(linha)
    print(linha)

with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as f:
    f.write("\n".join(resultado))

print("\n" + "=" * 60)
print("🎉 FINALIZADO COM SUCESSO!")
print(f"📁 Transcrição consolidada salva em: {ARQUIVO_SAIDA}")
print("=" * 60)

# Processamento concluído com sucesso: remove o checkpoint para não
# interferir em uma próxima execução (de um áudio diferente, por exemplo).
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print("🧹 Checkpoint removido (processamento concluído).")